In [53]:
import pandas as pd
from google.colab import files
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from google.colab import data_table
data_table.enable_dataframe_formatter()

df = None
nuevos_registros = []

# Valida si ya se cargó un archivo CSV antes de ejecutar cualquier análisis.
def validar_df_cargado():
    if df is None:
        print("\n Primero debes cargar un archivo CSV (Opción 1).\n")
        return False
    return True

# Función auxiliar interactiva para evitar errores al escribir el nombre de la columna.
def seleccionar_columna(dataframe, mensaje="Seleccione el número o nombre de la columna: "):
    columnas = list(dataframe.columns)
    print("\nColumnas disponibles:")
    for idx, col in enumerate(columnas, 1):
        print(f"  {idx}. {col}")

    ent = input(mensaje).strip()
    if ent.isdigit():
        idx = int(ent) - 1
        if 0 <= idx < len(columnas):
            return columnas[idx]

    # Si se puso el nombre en texto
    for col in columnas:
        if col.lower() == ent.lower():
            return col

    return None

while True:
  print("---------------------Menu Principal----------------------------------")
  print("1.	Cargar archivo CSV")
  print("2.	Limpiar el CSV")
  print("3.	Mostrar información del conjunto de datos")
  print("4.	Mostrar primeras y últimas filas")
  print("5.	Analizar tipos de datos")
  print("6.	Analizar valores nulos")
  print("7.	Analizar datos duplicados")
  print("8.	Obtener estadísticas descriptivas")
  print("9.	Filtrar o consultar datos")
  print("10. Realizar agrupaciones y operaciones")
  print("11. Generar representaciones gráficas")
  print("12. Realizar análisis adicional")
  print("13. Ingresar nuevos datos(Submenu)")
  print("14. Salir")
  print("----------------------------------------------------------------------")


  opcion=input("Selecione una opcion: ")

  if opcion=="1":
    try:
      # Seleccionar archivo
      print("\nSuba su archivo CSV:")
      archivo=files.upload()

      if archivo:
        nombre=next(iter(archivo))#Obtener el nombre
        df=pd.read_csv(nombre)#Leer el CSV
        print("\nArchivo cargado correctamente\n")
        print(f"Dimensiones del dataset: {df.shape[0]} filas y {df.shape[1]} columnas.")
        display(df)
      else:
        print("\nNo se seleccionó ningún archivo.")
    except Exception as e:
      print(f"\nOcurrió un error al cargar el archivo: {e}")

  elif opcion == "2":
        if validar_df_cargado():
            # Variables para acumular los cambios realizados
            duplicados_totales = 0
            nulos_totales = 0
            negativos_totales = 0
            texto_estandarizado = False

            while True:
                print("\n-----------------------SUBMENÚ DE LIMPIEZA -------------------")
                print("1. Eliminar duplicados")
                print("2. Eliminar valores nulos")
                print("3. Corregir valores negativos en columnas numéricas")
                print("4. Estandarizar textos (espacios y Formato Título)")
                print("5. Ejecutar limpieza automatica ")
                print("6. Mostrar resumen de cambios realizados")
                print("7. Regresar al menú principal")
                print("-------------------------------------------------------------------")

                sub_limp = input("Seleccione una opción de limpieza: ").strip()

                if sub_limp == "1":
                    duplicados = df.duplicated().sum()
                    df = df.drop_duplicates()
                    duplicados_totales += duplicados
                    print(f"\nFilas duplicadas eliminadas: {duplicados}")
                    print(f"Total de filas actuales: {len(df)}")

                elif sub_limp == "2":
                    nulos = df.isnull().sum().sum()
                    df = df.dropna()
                    nulos_totales += nulos
                    print(f"\n Filas con datos faltantes (nulos) eliminadas: {nulos}")
                    print(f"Total de filas actuales: {len(df)}")

                elif sub_limp == "3":
                    negativos_removidos = 0
                    cols_numericas = df.select_dtypes(include=['number']).columns
                    for col in cols_numericas:
                        cant_neg = (df[col] < 0).sum()
                        if cant_neg > 0:
                            negativos_removidos += cant_neg
                            df = df[df[col] >= 0]
                    negativos_totales += negativos_removidos
                    print(f"\nRegistros con valores negativos eliminados: {negativos_removidos}")
                    print(f"Total de filas actuales: {len(df)}")

                elif sub_limp == "4":
                    cols_texto = df.select_dtypes(include=['object']).columns
                    for col in cols_texto:
                        df[col] = df[col].astype(str).str.strip().str.title()
                    texto_estandarizado = True
                    print("\nCadenas de texto estandarizadas (espacios y mayúsculas/minúsculas).")

                elif sub_limp == "5":
                    print("\n--- Ejecutando limpieza total ---")
                    filas_inicio = len(df)

                    # 1. Duplicados
                    duplicados = df.duplicated().sum()
                    df = df.drop_duplicates()
                    duplicados_totales += duplicados

                    # 2. Nulos
                    nulos = df.isnull().sum().sum()
                    df = df.dropna()
                    nulos_totales += nulos

                    # 3. Negativos
                    negativos_removidos = 0
                    cols_numericas = df.select_dtypes(include=['number']).columns
                    for col in cols_numericas:
                        cant_neg = (df[col] < 0).sum()
                        if cant_neg > 0:
                            negativos_removidos += cant_neg
                            df = df[df[col] >= 0]
                    negativos_totales += negativos_removidos

                    # 4. Texto
                    cols_texto = df.select_dtypes(include=['object']).columns
                    for col in cols_texto:
                        df[col] = df[col].astype(str).str.strip().str.title()
                    texto_estandarizado = True

                    filas_fin = len(df)
                    filas_eliminadas = filas_inicio - filas_fin

                    print("Limpieza completa ejecutada con éxito.")
                    print(f"Total de filas procesadas: {filas_inicio} ➔ Filas finales: {filas_fin} (Se removieron {filas_eliminadas} registros).")

                    print("\nVista previa del dataset limpio:")
                    display(df)

                elif sub_limp == "6":
                    print("\n------------------ RESUMEN DE CAMBIOS APLICADOS -----------------------")
                    print(f" -- Filas duplicadas eliminadas: {duplicados_totales}")
                    print(f" -- Filas con datos faltantes (nulos) eliminadas: {nulos_totales}")
                    print(f" -- Registros con valores negativos eliminados: {negativos_totales}")

                    estado_texto = "Sí" if texto_estandarizado else "No"
                    print(f" -- Cadenas de texto estandarizadas: {estado_texto}")

                    print(f" Total de filas actuales en el dataset: {len(df)}")
                    print("-----------------------------------------------------------------------")

                    print("\nVista previa actual del dataset:")
                    display(df)

                elif sub_limp == "7":
                    print("\nRegresando al menú principal...")
                    break
                else:
                    print("\nOpción no válida. Intente de nuevo.")
  elif opcion=="3":
   if validar_df_cargado():
            print("\n--- INFORMACIÓN GENERAL DEL DATASET ---")
            print(f"Número total de filas: {df.shape[0]}")
            print(f"Número total de columnas: {df.shape[1]}")
            print(f"Nombres de las columnas:\n{list(df.columns)}")
            print("\n--- MATRIZ DE CORRELACIÓN ---")
            num_df = df.select_dtypes(include=['number'])

            if not num_df.empty:
                print("Matriz de Correlación entre variables numéricas:")
                display(num_df.corr().round(2))
            else:
                print("No existen columnas numéricas en el dataset para calcular la correlación.")

  elif opcion=="4":
    if validar_df_cargado():
      try:
        cant = int(input("¿Cuántas filas deseas visualizar? (por defecto 5): ") or 5)
        print(f"\n--- Primeras {cant} Filas ---")
        display(df.head(cant))
        print(f"\n--- Ultimas {cant} Filas ---")
        display(df.tail(cant))
      except ValueError:
        print("Ingresa un número entero válido.")

  elif opcion=="5":
    if validar_df_cargado():
      print("\n--- TIPOS DE DATOS POR COLUMNA ---")
      df_tipos = pd.DataFrame({
      'Columna': df.columns,
      'Tipo de Dato': df.dtypes.astype(str)
      })
      display(df_tipos)

  elif opcion=="6":
    if validar_df_cargado():
      print("\n--- ANÁLISIS DE VALORES NULOS ---")
      nulos = df.isnull().sum()
      porcentaje = (df.isnull().sum() / len(df)) * 100
      df_nulos = pd.DataFrame({
      'Valores Nulos': nulos,
      'Porcentaje (%)': porcentaje.round(2)
      })
    display(df_nulos)

  elif opcion=="7":
    if validar_df_cargado():
      duplicados = df.duplicated().sum()
      print(f"\n--- ANÁLISIS DE DUPLICADOS ---")
      print(f"Cantidad de filas completamente duplicadas: {duplicados}")
      if duplicados > 0:
        print("\nFilas duplicadas encontradas:")
        display(df[df.duplicated()])

  elif opcion=="8":
    if validar_df_cargado():
      print("\n--- ESTADÍSTICAS DESCRIPTIVAS (NUMÉRICAS) ---")
      display(df.describe().T)

      # Si existen columnas categóricas, mostrar su resumen
      cat_cols = df.select_dtypes(include=['object', 'category']).columns
      if len(cat_cols) > 0:
        print("\n--- ESTADÍSTICAS DESCRIPTIVAS (CATEGÓRICAS) ---")
        display(df.describe(include=['object', 'category']).T)

  elif opcion=="9":
    if validar_df_cargado():
      print("\n--- FILTRAR O CONSULTAR DATOS ---")
      columna = seleccionar_columna(df, "Selecciona la columna a filtrar: ")
      if columna:
        valor = input(f"Ingresa el valor a buscar en '{columna}': ").strip()
        if pd.api.types.is_numeric_dtype(df[columna]):
          try:
            valor_num = float(valor)
            resultado = df[df[columna] == valor_num]
          except ValueError:
            print("La columna es numérica, se realizará búsqueda textual del valor.")
            resultado = df[df[columna].astype(str).str.contains(valor, case=False, na=False)]
        else:
          resultado = df[df[columna].astype(str).str.contains(valor, case=False, na=False)]

        print(f"\nSe encontraron {len(resultado)} registro(s) coincidente(s):")
        display(resultado)
      else:
        print("Opción de columna no válida.")

  elif opcion=="10":
    if validar_df_cargado():
      print("\n--- AGRUPACIONES Y OPERACIONES ---")
      print(f"Columnas disponibles: {list(df.columns)}")
      col_agrupar = input("Ingresa la columna para agrupar: ").strip()
      col_calculo = input("Ingresa la columna numérica sobre la cual calcular: ").strip()

      if col_agrupar in df.columns and col_calculo in df.columns:
        if pd.api.types.is_numeric_dtype(df[col_calculo]):
          agrupado = df.groupby(col_agrupar)[col_calculo].agg(['count', 'mean', 'sum', 'min', 'max']).reset_index()
          print(f"\nResultados del agrupamiento por '{col_agrupar}':")
          display(agrupado)
        else:
          print(f"La columna '{col_calculo}' no es numérica.")
      else:
        print("Alguna de las columnas ingresadas no existe.")

  elif opcion=="11":
    if validar_df_cargado():
            print("\n--- GENERAR REPRESENTACIONES GRÁFICAS ---")
            print("1. Gráfico de Barras")
            print("2. Histograma ")
            print("3. Gráfico de Dispersión")
            print("4. Gráfico de Tendencia")
            print("5. Gráfico Boxplot (Detección de outliers)")
            print("6. Mostrar Mapa de calor(Comportamien y relacion de variables numericas)")
            print("7. Salir")



            sub_op = input("Selecciona una opción de gráfico: ").strip()

            if sub_op == "1":
                col = seleccionar_columna(df, "Selecciona la columna categórica: ")
                if col:
                    plt.figure(figsize=(8, 5))
                    df[col].value_counts().head(10).plot(kind='bar', color='skyblue', edgecolor='black')
                    plt.title(f"Top valores más frecuentes en '{col}'")
                    plt.xlabel(col)
                    plt.ylabel("Frecuencia")
                    plt.xticks(rotation=45)
                    plt.grid(axis='y', linestyle='--', alpha=0.7)
                    plt.tight_layout()
                    plt.show()

            elif sub_op == "2":
                col = seleccionar_columna(df, "Selecciona la columna numérica para el Histograma: ")
                if col and pd.api.types.is_numeric_dtype(df[col]):
                    datos = df[col].dropna()

                    q1 = datos.quantile(0.25)
                    q3 = datos.quantile(0.75)
                    iqr = q3 - q1
                    limite_superior = q3 + 1.5 * iqr
                    max_bigote = datos[datos <= limite_superior].max()

                    limite_x = max_bigote * 1.2
                    datos_filtrados = datos[datos <= limite_x]

                    fig, ax = plt.subplots(figsize=(11, 5), dpi=100)

                    ax.hist(
                        datos_filtrados,
                        bins=35,
                        color='#38b000',
                        edgecolor='#1b4332',
                        alpha=0.8
                    )

                    ax.set_title(f"Distribución de '{col}' (Enfoque en Rango Principal)", fontsize=13, pad=15, fontweight='bold')
                    ax.set_xlabel(col, fontsize=11, labelpad=10)
                    ax.set_ylabel("Frecuencia", fontsize=11, labelpad=10)

                    ax.set_xlim(left=0, right=limite_x)

                    # Marcas del eje x
                    import matplotlib.ticker as ticker
                    ax.xaxis.set_major_locator(ticker.MultipleLocator(100000))
                    ax.xaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
                    ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

                    plt.xticks(rotation=30, ha='right')

                    ax.grid(axis='y', linestyle='--', alpha=0.4)
                    for spine in ['top', 'right']:
                        ax.spines[spine].set_visible(False)
                    plt.tight_layout()
                    plt.show()
                else:
                    print("La columna seleccionada debe ser numérica.")

            elif sub_op == "3":
                col_x = seleccionar_columna(df, "Selecciona columna para Eje X: ")
                col_y = seleccionar_columna(df, "Selecciona columna para Eje Y: ")
                if col_x and col_y and pd.api.types.is_numeric_dtype(df[col_x]) and pd.api.types.is_numeric_dtype(df[col_y]):
                    plt.figure(figsize=(9, 5))
                    plt.scatter(df[col_x], df[col_y], alpha=0.5, color='coral', edgecolors='k')
                    plt.title(f"Relación entre {col_x} y {col_y}")
                    plt.xlabel(col_x)
                    plt.ylabel(col_y)
                    plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
                    plt.grid(True, linestyle='--', alpha=0.5)
                    plt.tight_layout()
                    plt.show()
                else:
                    print("Las columnas seleccionadas deben ser numéricas y válidas.")

            elif sub_op == "4":
                col_fecha = seleccionar_columna(df, "Selecciona la columna de Fecha: ")
                col_num = seleccionar_columna(df, "Selecciona la columna numérica a evaluar: ")
                if col_fecha and col_num and pd.api.types.is_numeric_dtype(df[col_num]):
                    df_temp = df.copy()
                    df_temp[col_fecha] = pd.to_datetime(df_temp[col_fecha], errors='coerce')
                    df_temp = df_temp.dropna(subset=[col_fecha]).sort_values(col_fecha)

                    agrupado_tiempo = df_temp.groupby(col_fecha)[col_num].sum().reset_index()

                    plt.figure(figsize=(16, 7), dpi=120)

                    plt.plot(
                        agrupado_tiempo[col_fecha],
                        agrupado_tiempo[col_num],
                        marker='o',
                        markersize=4,
                        color='purple',
                        linewidth=1.2
                    )

                    plt.title(f"Tendencia Temporal de {col_num}", fontsize=14, pad=15, fontweight='bold')
                    plt.xlabel(col_fecha, fontsize=11, labelpad=10)
                    plt.ylabel(f"Suma de {col_num}", fontsize=11, labelpad=10)

                    # Formato numérico de Y
                    plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

                    # Formato de fechas original (AAAA-MM) con rotación a 45 grados
                    plt.gca().xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m'))

                    plt.xticks(rotation=45, ha='right', fontsize=10)
                    plt.yticks(fontsize=10)
                    plt.grid(True, linestyle='--', alpha=0.5)
                    plt.tight_layout()
                    plt.show()
                else:
                    print("Verifica que la columna numérica sea válida.")

            elif sub_op == "5":
                col = seleccionar_columna(df, "Selecciona la columna numérica para el Boxplot: ")
                if col and pd.api.types.is_numeric_dtype(df[col]):
                    datos = df[col].dropna()

                    #Calculo de quartiles
                    q1 = datos.quantile(0.25)
                    mediana = datos.median()
                    q3 = datos.quantile(0.75)
                    iqr = q3 - q1

                    limite_inferior = q1 - 1.5 * iqr
                    limite_superior = q3 + 1.5 * iqr

                    min_bigote = datos[datos >= limite_inferior].min()
                    max_bigote = datos[datos <= limite_superior].max()
                    cant_outliers = ((datos < limite_inferior) | (datos > limite_superior)).sum()

                    # Grafico
                    fig, ax = plt.subplots(figsize=(10, 6), dpi=100)

                    bp = ax.boxplot(
                        datos,
                        patch_artist=True,
                        boxprops=dict(facecolor='#a8dadc', color='#1d3557', linewidth=1.5),
                        medianprops=dict(color='#e63946', linewidth=2),
                        whiskerprops=dict(color='#1d3557', linewidth=1.5),
                        capprops=dict(color='#1d3557', linewidth=1.5),
                        flierprops=dict(marker='o', markerfacecolor='#e63946', markeredgecolor='none', alpha=0.5, markersize=5)
                    )

                    #Cuadro con estadisticas
                    texto_resumen = (
                        "RESUMEN ESTADÍSTICO\n"
                        "─────────────────────────────\n"
                        f"Máx (Bigote) : {max_bigote:,.2f}\n"
                        f"Q3 (75%)     : {q3:,.2f}\n"
                        f"Mediana (50%): {mediana:,.2f}\n"
                        f"Q1 (25%)     : {q1:,.2f}\n"
                        f"Mín (Bigote) : {min_bigote:,.2f}\n"
                        "─────────────────────────────\n"
                        f"Rango Intercuartil: {iqr:,.2f}\n"
                        f"Atípicos (Outliers): {cant_outliers} datos"
                    )

                    # Propiedades visuales de la tarjeta
                    props_caja = dict(
                        boxstyle='round,pad=0.8',
                        facecolor='#f8fafc',
                        edgecolor='#cbd5e1',
                        alpha=0.95
                    )

                    ax.text(
                        0.97, 0.95, texto_resumen,
                        transform=ax.transAxes,
                        fontsize=9,
                        verticalalignment='top',
                        horizontalalignment='right',
                        fontfamily='monospace',
                        color='#1e293b',
                        bbox=props_caja
                    )


                    plt.title(f"Gráfico de Caja (Boxplot) de '{col}'", fontsize=13, pad=15, fontweight='bold')
                    plt.ylabel(col, fontsize=11)
                    plt.xticks([1], [col], fontsize=10)

                    plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
                    plt.grid(axis='y', linestyle='--', alpha=0.5)
                    plt.tight_layout()
                    plt.show()
                else:
                    print("La columna seleccionada debe ser numérica.")

            elif sub_op == "6":
                num_df = df.select_dtypes(include=['number'])
                if num_df.shape[1] >= 2:
                    corr = num_df.corr()
                    plt.figure(figsize=(8, 6))
                    plt.imshow(corr, cmap='coolwarm', interpolation='none')
                    plt.colorbar(label='Coeficiente de Correlación')

                    for i in range(len(corr)):
                        for j in range(len(corr)):
                            plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha='center', va='center', color='black')

                    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45)
                    plt.yticks(range(len(corr.columns)), corr.columns)
                    plt.title("Mapa de Calor de Correlación (Heatmap)")
                    plt.tight_layout()
                    plt.show()
                else:
                    print("Se necesitan al menos 2 columnas numéricas para el mapa de calor.")

            else:
                print("Opción de gráfico no válida.")

  elif opcion=="12":
    print("""
--------------------------------------------------------------------------------
              ANALISIS ADICIONAL(Storytelling)
--------------------------------------------------------------------------------

Al analizar los datos de ventas de la tienda de tecnología,
lo primero que encontramos fue que la información necesitaba
un proceso de revisión y limpieza antes de utilizarse para
tomar decisiones. Durante la exploración se identificaron
datos faltantes, registros duplicados, valores inconsistentes
y algunos valores negativos que podían afectar los resultados.
Después de aplicar el proceso de limpieza fue posible trabajar
con una base de datos más consistente.

Una vez preparados los datos, encontramos que algunas
categorías presentan un mayor movimiento que otras. Esto
permite identificar los grupos de productos que tienen una
mayor presencia dentro de las operaciones y que, por lo tanto,
pueden requerir mayor atención en la planificación del
inventario.

Al analizar los productos individuales también fue posible
identificar los diez productos que aparecen con mayor
frecuencia en las ventas. Estos productos representan una
parte importante de la actividad comercial y pueden ser
considerados productos que requieren especial atención para
evitar problemas de disponibilidad.

El comportamiento de las ventas también cambia a través del
tiempo. El análisis temporal permite identificar períodos de
mayor y menor actividad, información que puede ser utilizada
para anticipar necesidades de inventario, planificar compras
y preparar estrategias comerciales para diferentes momentos
del año.

Desde el punto de vista del equipo comercial, los vendedores
presentan diferentes niveles de actividad dentro de los
registros analizados. Esta información permite identificar
diferencias que pueden utilizarse como punto de partida para
evaluar el desempeño, establecer metas y detectar buenas
prácticas dentro del equipo.

Otro aspecto importante aparece al analizar los precios.
La distribución de precio_unitario muestra una concentración
de valores en determinados rangos, pero también presenta
valores considerablemente alejados del comportamiento general.
El análisis mediante el boxplot identificó 55 valores atípicos.
Estos registros deben revisarse para determinar si representan
productos de alto valor o posibles errores de registro.

Finalmente, al estudiar la relación entre cantidad y
precio_unitario, no se encontró una relación lineal fuerte
entre ambas variables. Esto significa que dentro de los datos
analizados no podemos asumir que un producto con un precio
mayor necesariamente se venda en una cantidad menor, ni que
un precio menor implique automáticamente una mayor cantidad
vendida.

En conjunto, los datos cuentan una historia importante para
el negocio: existen productos y categorías con mayor actividad,
existen diferencias entre vendedores y períodos de venta,
y también existen valores que requieren una revisión adicional.
Por esta razón, el análisis exploratorio no solamente permite
describir los datos, sino también convertirlos en información
que puede apoyar decisiones relacionadas con inventario,
productos, precios y actividad comercial.

--------------------------------------------------------------------------------
                    FIN DEL STORYTELLING

""")
  elif opcion=="13":
    while True:
          print("\n---------------------------- SUBMENÚ -------------------------------")
          print("1. Ingresar un nuevo registro")
          print("2. Mostrar registros ingresados")
          print("3. Guardar los nuevos datos en el dataset")
          print("4. Exportar y descargar CSV actualizado")
          print("5. Ver el Readme")
          print("6. Regresar al menú principal")
          print("-----------------------------------------------------------------------")

          sub_opcion = input("Seleccione una opción: ").strip()

          if sub_opcion == "1":
              if validar_df_cargado():
                  print("\n--- INGRESAR NUEVO REGISTRO ---")
                  nuevo_dict = {}
                  for col in df.columns:
                      val = input(f"Ingrese valor para '{col}': ").strip()
                      # Intenta convertir a tipo numérico si aplica
                      try:
                          if "." in val:
                              val = float(val)
                          else:
                              val = int(val)
                      except ValueError:
                          pass  # Se queda como string si no es convertible
                      nuevo_dict[col] = val

                  nuevos_registros.append(nuevo_dict)
                  print("\n Registro agregado temporalmente a la cola de pendientes.")

          elif sub_opcion == "2":
              print("\n--- REGISTROS NUEVOS PENDIENTES DE UNIR ---")
              if nuevos_registros:
                  df_nuevos = pd.DataFrame(nuevos_registros)
                  display(df_nuevos)
              else:
                  print("No se ha ingresado ningún registro nuevo aún en esta sesión.")

          elif sub_opcion == "3":
              if nuevos_registros:
                  df_nuevos = pd.DataFrame(nuevos_registros)
                  df = pd.concat([df, df_nuevos], ignore_index=True)
                  nuevos_registros.clear()
                  print("\n[OK] Los nuevos registros fueron incorporados exitosamente al dataset en memoria.")
                  print(f"Total de filas actuales: {len(df)}")
              else:
                  print("\n[!] No hay nuevos registros pendientes para incorporar.")

          elif sub_opcion == "4":
              if validar_df_cargado():
                  nombre_salida = "ventas_actualizado.csv"
                  # Guarda el DataFrame actual en un archivo CSV local en Colab
                  df.to_csv(nombre_salida, index=False, encoding='utf-8-sig')
                  print(f"\n Archivo '{nombre_salida}' generado con éxito.")

                  # Inicia la descarga automática en el navegador del usuario
                  files.download(nombre_salida)
                  print("El archivo se descargara una vez se reinice el programa")


          elif sub_opcion == "5":
              print("""
              # Proyecto Final - Análisis Exploratorio de Datos (EDA)

              ## Análisis de ventas de una tienda de tecnología

              Proyecto desarrollado para aplicar técnicas de Análisis Exploratorio de Datos (EDA) sobre un conjunto de datos de ventas de una tienda de tecnología.

              El programa permite cargar un archivo CSV, explorar su estructura, identificar problemas de calidad, limpiar los datos, realizar cálculos y agrupaciones, generar visualizaciones y obtener conclusiones orientadas al análisis del negocio.

              ---
              ## Objetivo del proyecto
              El objetivo principal es utilizar Python y herramientas de análisis de datos para transformar un conjunto de registros de ventas en información útil para comprender el comportamiento del negocio.
              El análisis busca responder preguntas relacionadas con:
              - Categorías con mayor movimiento.
              - Productos con mayor frecuencia de venta.
              - Comportamiento de las ventas a través del tiempo.
              - Nivel de actividad de los vendedores.
              - Distribución de los precios.
              - Valores atípicos.
              - Relación entre cantidad y precio.
              - Métodos de pago utilizados.
              - Comportamiento general de las ventas.
              ---
              ## Tecnologías utilizadas
              El proyecto fue desarrollado utilizando:
              - Python
              - Pandas
              - Matplotlib
              - Tkinter
              - Google Colab
              """)
              break

          elif sub_opcion == "6":
              print("\nRegresando al menú principal...")
              break

          else:
              print("\nOpción incorrecta, intente de nuevo.")

  elif opcion=="14":
    print("Gracias por usar el programa")
    break

  else:
    print("Opcion incorrecta")




---------------------Menu Principal----------------------------------
1.	Cargar archivo CSV
2.	Limpiar el CSV
3.	Mostrar información del conjunto de datos
4.	Mostrar primeras y últimas filas
5.	Analizar tipos de datos
6.	Analizar valores nulos
7.	Analizar datos duplicados
8.	Obtener estadísticas descriptivas
9.	Filtrar o consultar datos
10. Realizar agrupaciones y operaciones
11. Generar representaciones gráficas
12. Realizar análisis adicional
13. Ingresar nuevos datos(Submenu)
14. Salir
----------------------------------------------------------------------
Selecione una opcion: 14
Gracias por usar el programa


# Proyecto Final - Análisis Exploratorio de Datos (EDA)

## Análisis de ventas de una tienda de tecnología

Proyecto desarrollado para aplicar técnicas de Análisis Exploratorio de Datos (EDA) sobre un conjunto de datos de ventas de una tienda de tecnología.

El programa permite cargar un archivo CSV, explorar su estructura, identificar problemas de calidad, limpiar los datos, realizar cálculos y agrupaciones, generar visualizaciones y obtener conclusiones orientadas al análisis del negocio.

---

## Objetivo del proyecto

El objetivo principal es utilizar Python y herramientas de análisis de datos para transformar un conjunto de registros de ventas en información útil para comprender el comportamiento del negocio.

El análisis busca responder preguntas relacionadas con:

- Categorías con mayor movimiento.
- Productos con mayor frecuencia de venta.
- Comportamiento de las ventas a través del tiempo.
- Nivel de actividad de los vendedores.
- Distribución de los precios.
- Valores atípicos.
- Relación entre cantidad y precio.
- Métodos de pago utilizados.
- Comportamiento general de las ventas.

---

## Tecnologías utilizadas

El proyecto fue desarrollado utilizando:

- Python
- Pandas
- Matplotlib
- Tkinter
- Google Colab